# 2. Feedback Linearizing Controller

This notebook introduces the *control point* and the path-following controller based on feedback linearization.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using LinkedMasses

## Control Point

Let $p_0 = [x_0, y_0]$ be the position of the ASV, $\theta$ the angle of the link, and $L$ the length of the cable. The control point is defined as

$$ p_\varepsilon = p_0 + \varepsilon L [\cos\theta, \sin\theta] $$

where $\varepsilon \in (0, 1)$ is a constant.

The goal of the controller is to steer the point $p_\varepsilon$ to a desired path.
The desired path is parametrized by a smooth function $p_p : \mathbb{R} \mapsto \mathbb{R}^2$.

## Simulation State

The state of the simulation consists of seven states - the state of the towed system and the path parameter. For ease of use, there is a structure `SimulationState` and functions `pack_state` and `unpack_state`:

In [ ]:
@doc SimulationState

In [ ]:
@doc pack_state

In [ ]:
@doc unpack_state

The control scheme consists of line-of-sight guidance and a feedback linearizing controller.

## Line-of-Sight Guidance

LOS guidance has two parameters - desired speed, $U$, and lookahead distance, $\Delta$. The parameters are encapsulated in `LOSParameters`:

In [ ]:
@doc LOSParameters

LOS guidance has two outputs: desired velocity vector $v_\varepsilon^{ref}$, and derivative of the path parameter, $\dot{s}$.

## Feedback-Linearizing Controller

Define $\tilde{v}_\varepsilon = v_\varepsilon - v_\varepsilon^{ref}$.
If all parameters of the system are known, it is possible to design a control law such that

$$ \dot{\tilde{v}}_\varepsilon = -k_v \tilde{v}_\varepsilon $$

where $k_v$ is a positive constant. To simulate the system with the feedback-linearizing controller, we need to create an instance of `LinearizingController`:

In [ ]:
@doc LinearizingController

To simulate the system, use `closed_loop_ode!`

In [ ]:
@doc closed_loop_ode!

## Example

Let us take the linked-masses model from the previous notebook with a non-zero ocean current:

In [ ]:
V_c = [-0.2, 0.0]

m0 = 200.0
m = 40.0
c0 = 250.0
c = 50.0
L = 10.0

params = LinkedMassesParameters(L, m0, m, c0, c)

For a desired path, we choose a circle with a radius of 10 meters. The origin of the circle is at $[2, 12]$, and the initial phase is $-\pi/2$.

In [ ]:
R = 10.0
ϕ0 = -π / 2
p_origin = [2.0, 12.0]
path_circle(s) = p_origin + R * [cos(ϕ0 + s), sin(ϕ0 + s)]

Since the WAM-V is relatively slow, we choose a desired speed of 1 m/s and a lookahead distance of 5 m.

In [ ]:
U = 1.0
Δ = 5.0
los = LOSParameters(U, Δ)

Finally, we set the velocity gain to 0.1 and the control point coefficient to 0.7:

In [ ]:
k_v = 0.1
ε = 0.7

ctrl = LinearizingController(params, los, path_circle, k_v, ε, V_c)

To visualize the data, we can use the `make_gif` function:

In [ ]:
@doc make_gif

In [ ]:
using DifferentialEquations
T_stop = 2π * R / U

# Initial state: both masses at rest, first mass at origin, second mass east of it
p0_init = [0.0, 0.0]
θ_init = -π / 2
v0_init = [0.0, 0.0]
ω_init = 0.0
s_init = 0.0
init_state = SimulationState(p0_init, θ_init, v0_init, ω_init, s_init)

prob = ODEProblem(closed_loop_ode!, pack_state(init_state), (0.0, T_stop), ctrl)
sol = solve(prob, Tsit5())

# GIF parameters
fps = 20
spd = 3.0
filename = "linearizing_controller"
directory = joinpath(@__DIR__, "gifs")
mkpath(directory)
make_gif(sol, ctrl, filename; fps=fps, speed=spd, output_dir=directory)